# 수정주가를 채우다 — 외부가 안 주는 4년을 어떻게 메웠나

> `notebooks/02-품질·전처리/04.수정주가를채우다.ipynb` · 2026-09-02 작성 · **2026-09-03 보강** · 이동원
> 마이그레이션 v9 · 이슈 [#51](https://github.com/devlee328288/Alpha_Stack/issues/51) · [#80](https://github.com/devlee328288/Alpha_Stack/issues/80)

---

## 이 노트북이 답하는 것

> **"삼성전자가 하루 만에 −98% 폭락한 것으로 읽히는데, 어떻게 고쳤나?"**

`daily_price.close` 는 KRX 원문 그대로라 **액면분할이 조정돼 있지 않습니다.**
분할일에 가격이 그대로 뚝 떨어지므로, `close` 로 수익률을 계산하면 폭락으로 읽힙니다.

이 문제를 **다 고쳤습니다.** 다만 고치는 과정에서 계획을 두 번 바꿔야 했고,
고친 뒤에도 **한 곳이 더 남아 있었습니다.** 그 셋이 이 노트북의 내용입니다.

| 무엇을 하려 했나 | 무엇이 막았나 | 어떻게 했나 |
|---|---|---|
| FDR 수정주가를 받아 싣는다 | FDR 이 **최근 3,000거래일만** 준다 | 그 앞은 우리 조정계수로 이어 붙였다 |
| FDR 의 시·고·저·종가를 그대로 싣는다 | FDR 이 네 칸을 **따로 반올림**해 `고가 < 종가` 가 된다 | 배율 하나를 넷에 똑같이 곱했다 |
| 팀에 `adj_*` 를 쓰라고 안내한다 | **정작 우리 반출이 원가격을 보고 있었다** (신장환 팀원 발견) | 가격 기준을 한 자리로 모으고 재반출했다 |

결과부터 적으면 — **9,223,644행 전부(100%)** 채웠고, `fdr` 81.6% · `chain` 18.4% 입니다.
(9~10장의 배포본 통계에서는 `fdr` 78.4% · `chain` 21.6% 로 나옵니다. 배포본은 홀드아웃을
덜어낸 개발구간 7,888,945행이고, 잘려 나간 최근 구간이 바로 외부가 주는 구간이라
`fdr` 비중이 내려갑니다. 같은 자료를 다른 창으로 본 값입니다.)

---

## 0. 준비

In [1]:
import sqlite3
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import pandas as pd

from common.paths import krx_db_path

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

conn = sqlite3.connect(f"file:{krx_db_path().as_posix()}?mode=ro", uri=True)
print("DB :", krx_db_path())
print("스키마 v", conn.execute("PRAGMA user_version").fetchone()[0])

DB : C:\Users\kik32\workspace\EST-Camp-AI-Quant\team_project\Alpha_Stack\data\krx_cache.db
스키마 v 9


---

## 1. 문제 — 삼성전자 2018-05-04

액면분할 50:1 이 일어난 날입니다. **원문을 그대로** 뽑아 보겠습니다.

In [2]:
삼성 = pd.read_sql_query(
    """SELECT bas_dd, open, high, low, close, change_rate, volume, listed_shares,
              adj_open, adj_high, adj_low, adj_close, adj_source
       FROM daily_price WHERE code = '005930'
         AND bas_dd BETWEEN '20180426' AND '20180508' ORDER BY bas_dd""",
    conn)
삼성[["bas_dd", "open", "high", "low", "close", "change_rate", "volume", "listed_shares"]]

,bas_dd,open,high,low,close,change_rate,volume,listed_shares
0,20180426,2521000,2608000,2520000,2607000,3.45,360931,128386494
1,20180427,2669000,2682000,2622000,2650000,1.65,606216,128386494
2,20180430,0,0,0,2650000,0.00,0,128386494
3,20180502,0,0,0,2650000,0.00,0,128386494
4,20180503,0,0,0,2650000,0.00,0,128386494
5,20180504,53000,53900,51800,51900,-2.08,39565391,6419324700
6,20180508,52600,53200,51900,52600,1.35,23104720,6419324700


세 가지가 한꺼번에 보입니다.

1. **20180504 에 종가가 2,650,000 → 51,900** 으로 떨어집니다. 그런데 `change_rate` 는 **−2.08%** 입니다.
2. **20180430·0502·0503 은 시·고·저가가 0** 입니다 — 거래정지입니다.
   KRX 는 주권 교체 때문에 분할 전에 반드시 거래를 정지시킵니다.
3. **상장주식수가 128,386,494 → 6,419,324,700** 으로 정확히 50배가 됩니다.

`close` 로 계산한 수익률과 KRX 가 알려 주는 실제 등락률을 나란히 놓아 보겠습니다.

In [3]:
원가격수익률 = 51900 / 2650000 - 1
print(f"close 로 계산     : {원가격수익률 * 100:+.2f}%")
print(f"KRX change_rate  : {삼성.loc[삼성.bas_dd == '20180504', 'change_rate'].iloc[0]:+.2f}%")
print()
print("→ 같은 날에 대한 두 값이 96%p 차이납니다. close 가 미조정 원가격이라 그렇습니다.")

close 로 계산     : -98.04%
KRX change_rate  : -2.08%

→ 같은 날에 대한 두 값이 96%p 차이납니다. close 가 미조정 원가격이라 그렇습니다.


### 규모 — 우리 자료 전체에서

분할·병합은 삼성전자 하나의 일이 아닙니다. **주식수 배율과 가격 배율이 서로 역수인 날**을
분할·병합으로 판정해 전수로 셌습니다.

In [4]:
규모 = pd.read_sql_query(
    """WITH seq AS (
         SELECT code, bas_dd, close, listed_shares,
                LAG(close) OVER (PARTITION BY code ORDER BY bas_dd) AS 앞종가,
                LAG(listed_shares) OVER (PARTITION BY code ORDER BY bas_dd) AS 앞주식수
         FROM daily_price WHERE close > 0 AND listed_shares > 0)
       SELECT COUNT(*) AS 이벤트, COUNT(DISTINCT code) AS 종목
       FROM seq
       WHERE 앞종가 > 0 AND 앞주식수 > 0
         AND ABS((CAST(listed_shares AS REAL) / 앞주식수)
                 * (CAST(close AS REAL) / 앞종가) - 1) < 0.10
         AND (CAST(listed_shares AS REAL) / 앞주식수 > 1.5
              OR CAST(listed_shares AS REAL) / 앞주식수 < 1 / 1.5)""",
    conn)
전체종목 = conn.execute("SELECT COUNT(DISTINCT code) FROM daily_price").fetchone()[0]
print(f"분할·병합 이벤트 {규모.이벤트[0]:,}건 · {규모.종목[0]:,}종 "
      f"(전체 {전체종목:,}종의 {규모.종목[0] / 전체종목 * 100:.1f}%)")

분할·병합 이벤트 665건 · 538종 (전체 3,677종의 14.6%)


---

## 2. 첫 번째 벽 — FDR 이 2010년을 안 준다

FinanceDataReader(MIT · 0.9.202)가 수정주가를 줍니다. 그런데 **범위가 짧습니다.**

`start` 를 2010-01-01 로 줘도 2014-06-13 부터만 옵니다. FDR 의 한국 주식 경로는
네이버 `fchart` 이고 **그 서버가 3,000건에서 자릅니다.** FDR 코드는 이미 `count=6000` 을
보내고 있는데도 그렇습니다 — 직접 확인한 것이 아래입니다.

In [5]:
import re

import requests

for cnt in (3000, 6000, 9000):
    url = ("https://fchart.stock.naver.com/sise.nhn"
           f"?timeframe=day&count={cnt}&requestType=0&symbol=005930")
    items = re.findall(r'<item data="(.*?)" />', requests.get(url, timeout=30).text)
    print(f"count={cnt:>4} → {len(items):,}건 · 첫날 {items[0].split('|')[0]}")

count=3000 → 3,000건 · 첫날 20140616
count=6000 → 3,000건 · 첫날 20140616


count=9000 → 3,000건 · 첫날 20140616


요청을 아무리 키워도 **3,000건**입니다. 우리가 요청을 잘못한 것이 아니라 서버가 자릅니다.

pykrx 의 `get_market_ohlcv(..., adjusted=True)` 도 같은 네이버 경로라 결과가 같습니다.
그래서 **2010-01-04 ~ 2014-06-12 구간에는 외부 수정주가가 없습니다.** 그 크기를 재 보겠습니다.

In [6]:
구멍 = pd.read_sql_query(
    """SELECT
         (SELECT COUNT(*) FROM daily_price) AS 전체,
         (SELECT COUNT(*) FROM daily_price WHERE bas_dd < '20140613') AS 이전,
         (SELECT COUNT(DISTINCT bas_dd) FROM daily_price
            WHERE bas_dd < '20140613') AS 이전거래일""",
    conn)
비율 = 구멍.이전[0] / 구멍.전체[0] * 100
print(f"20140613 이전 : {구멍.이전[0]:,}행 · {구멍.이전거래일[0]:,}거래일 ({비율:.1f}%)")
print()
print("홀드아웃이 20240901 이므로 이 구멍은 **전부 학습구간 안**입니다.")
print("미조정으로 두면 #51 이 지적한 -98% 가 그 구간에 그대로 남습니다.")

20140613 이전 : 2,146,042행 · 1,103거래일 (23.3%)

홀드아웃이 20240901 이므로 이 구멍은 **전부 학습구간 안**입니다.
미조정으로 두면 #51 이 지적한 -98% 가 그 구간에 그대로 남습니다.


### 그래서 — 앵커를 놓고 뒤로 이어 붙였다

FDR 이 닿는 **가장 이른 날**을 앵커로 삼습니다. 그 날의 배율을 FDR 이 알려 주므로,
거기서부터 우리 조정계수를 곱해 과거로 내려갑니다.

```
20100104 ────────────── 20140612 │ 20140613 ────────── 20260901
  chain (계수로 뒤로 이어 붙임)   │      fdr (외부 실측)
                                 ↑
                          앵커. 이 날의 배율을 FDR 이 알려 준다
```

배율의 정의는 `scale[i] = adj_close[i] / close[i]` 이고, 옮기는 규칙은 두 줄입니다.

```
뒤로 (과거 방향)   scale[i] = scale[i+1] × factor[i+1]
앞으로 (미래 방향) scale[i] = scale[i-1] ÷ factor[i]
```

`factor` 는 그 날 일어난 조정이고 **그 앞의 행들에** 적용되므로, 과거로 갈 때 곱하고
미래로 갈 때 나눕니다. 계수 계산 자체는 `common/corporate_actions.py` 에 이미 있었습니다
(평상일은 기준가, 재개일은 상장주식수 배율 — 왜 둘로 나뉘는지는 그 파일 주석에 있습니다).

**배율은 `Fraction` 으로 옮깁니다.** 1/50 같은 계수가 수천 행에 걸쳐 곱해지므로
부동소수로 누적하면 반올림 잡음이 되돌아옵니다.

### 🔴 그런데 이 계산이 맞는지 어떻게 아나

2010~2014 구간은 **대조할 외부 자료가 없습니다.** 그게 애초에 이어 붙인 이유입니다.

그래서 이렇게 쟀습니다 — **FDR 을 일부러 앵커 하루만 남기고 지운 뒤**, 나머지 2,999일을
전부 우리 계산으로 채웁니다. 그리고 진짜 FDR 값과 비교합니다.
그 차이가 곧 **2010~2014 에 우리가 넣은 값의 오차 상한**입니다.

In [7]:
from ingest.clients import fdr_data
from ingest.store import adj_price

결과 = []
표본 = [("005930", "삼성전자", "20180504", 50),
        ("035420", "NAVER", "20181012", 5),
        ("035720", "카카오", "20210415", 5)]
for code, name, split_dd, ratio in 표본:
    rows = adj_price.load_rows(conn, code)
    adjusted = fdr_data.fetch_adjusted(code)
    # 🔴 앵커를 날짜로 못 박지 않는다. FDR 은 최근 3,000거래일만 주는데 그 창이
    #    오늘이 아니라 **종목의 마지막 거래일**에 걸린다. 날짜를 적어 두면 하루만
    #    지나도 창 밖으로 밀려나 이 셀이 KeyError 로 죽는다 — 2026-09-03 에 실제로
    #    났다(20140613). 그래서 "FDR 이 주는 가장 이른 날" 을 그때그때 앵커로 잡는다.
    앵커 = min(adjusted)
    ours = {r[-1]: r[3] for r in adj_price.build_rows(rows, {앵커: adjusted[앵커]})}
    차이 = [abs(ours[d] - v["adj_close"]) / v["adj_close"]
            for d, v in adjusted.items() if v["adj_close"] and ours.get(d)]
    차이.sort()
    결과.append({"종목": f"{name}({code})", "앵커": 앵커, "분할일": split_dd,
                 "대조일수": len(차이),
                 "중앙(%)": 차이[len(차이) // 2] * 100,
                 "최대(%)": 차이[-1] * 100, "공지분할": f"{ratio}:1"})
pd.DataFrame(결과)

,종목,앵커,분할일,대조일수,중앙(%),최대(%),공지분할
0,삼성전자(005930),20140616,20180504,2998,0.000000,0.000000,50:1
1,NAVER(035420),20140616,20181012,2998,0.141688,0.141688,5:1
2,카카오(035720),20140616,20210415,2998,0.040223,0.397223,5:1


**최대 0.397%.** 12년치를 앵커 하나에서 이어 붙였는데 그 정도로 재현됩니다.

이 값은 실행할 때마다 조금씩 움직입니다. FDR 의 3,000거래일 창이 날마다 하루씩
밀리면서 앵커가 바뀌기 때문입니다 — 2026-09-02 에는 `20140613` 이었고 오늘은
`20140616` 입니다. 그때는 0.39% 였습니다. **0.4% 안쪽이라는 것이 요지**이고,
소수 셋째 자리는 그날의 앵커가 정합니다.

NAVER 를 보면 중앙값과 최대값이 같습니다(0.142%). 이건 **상수 배율 차이**라는 뜻이고,
수익률은 비율이라 **상수 배율은 약분돼 사라집니다.** 즉 학습에 쓰는 값에는 영향이 없습니다.

---

## 3. 두 번째 벽 — FDR 이 고가를 종가보다 낮게 준다

처음에는 FDR 이 준 네 칸을 그대로 실었습니다. 그랬더니 표본 3종에서만
**고저 관계 위반 68행**이 나왔습니다. 전부 `fdr` 출처였고, 원가격 위반은 0행이었습니다.

In [8]:
사례 = pd.read_sql_query(
    """SELECT bas_dd, open, high, low, close, adj_open, adj_high, adj_low, adj_close
       FROM daily_price WHERE code = '005930' AND bas_dd = '20150127'""", conn)
print("원문 :", 사례[["open", "high", "low", "close"]].to_dict("records")[0])
print()
print("FDR 이 그대로 주던 값 : adj_high=27,999  adj_close=28,000   ← 고가 < 종가")
print("지금 우리가 싣는 값   :",
      {k: v for k, v in 사례[["adj_open", "adj_high", "adj_low", "adj_close"]]
       .to_dict("records")[0].items()})

원문 : {'open': 1375000, 'high': 1400000, 'low': 1374000, 'close': 1400000}

FDR 이 그대로 주던 값 : adj_high=27,999  adj_close=28,000   ← 고가 < 종가
지금 우리가 싣는 값   : {'adj_open': 27500.0, 'adj_high': 28000.0, 'adj_low': 27480.0, 'adj_close': 28000.0}


원문에서는 `high == close == 1,400,000` 으로 **같습니다.**
FDR 은 네 칸을 각각 따로 반올림하기 때문에 `1,400,000 / 50 = 28,000` 이 되어야 할 고가가
27,999 로 내려앉았습니다.

`true_range`·`parkinson_20` 처럼 고저 폭을 쓰는 피처는 여기서 **음수**를 뱉습니다.

**고친 방법은 단순합니다** — 배율 하나를 네 칸에 똑같이 곱합니다.
하루 안의 시·고·저·종가는 같은 스케일이므로, 이렇게 하면 그 날의 고저 폭과 시종 관계가
**정확히 보존됩니다.** 뒤집히던 칸만 제자리를 찾고 나머지는 FDR 값과 그대로 일치합니다.

In [9]:
위반 = conn.execute(
    """SELECT COUNT(*) FROM daily_price
       WHERE adj_high IS NOT NULL AND adj_low IS NOT NULL AND adj_close IS NOT NULL
         AND (adj_close > adj_high + 0.01 OR adj_close < adj_low - 0.01)""").fetchone()[0]
print(f"전체 9,223,644행에서 고저 관계 위반: {위반:,}행")

전체 9,223,644행에서 고저 관계 위반: 0행


---

## 4. 정지일 — 0 을 가격으로 실으면 안 된다

거래정지일은 `open = high = low = 0` 이고 종가만 직전 값을 물고 있습니다.
FDR 도 정확히 같은 모양으로 줍니다.

0 에 배율을 곱하면 0 이 되어 **"그 날 가격이 0원이었다"** 가 됩니다. 수익률은 −100% 가 되고,
고저 검사도 `0 ≤ 0 ≤ 0` 이라 통과해 버립니다. 그래서 **시·고·저가는 비우고 종가만 채웁니다.**

In [10]:
삼성[["bas_dd", "open", "high", "low", "close",
      "adj_open", "adj_high", "adj_low", "adj_close", "adj_source"]]

,bas_dd,open,high,low,close,adj_open,adj_high,adj_low,adj_close,adj_source
0,20180426,2521000,2608000,2520000,2607000,50420.0,52160.0,50400.0,52140.0,fdr
1,20180427,2669000,2682000,2622000,2650000,53380.0,53640.0,52440.0,53000.0,fdr
2,20180430,0,0,0,2650000,NaN,NaN,NaN,53000.0,fdr
3,20180502,0,0,0,2650000,NaN,NaN,NaN,53000.0,fdr
4,20180503,0,0,0,2650000,NaN,NaN,NaN,53000.0,fdr
5,20180504,53000,53900,51800,51900,53000.0,53900.0,51800.0,51900.0,fdr
6,20180508,52600,53200,51900,52600,52600.0,53200.0,51900.0,52600.0,fdr


20180430·0502·0503 의 `adj_open`·`adj_high`·`adj_low` 가 `None` 이고 `adj_close` 는 살아 있습니다.
그 종가가 **재개일 등락률의 기준**이 되기 때문에 버리면 안 됩니다.

이제 분할일 수익률이 어떻게 됐는지 보겠습니다.

In [11]:
전 = 삼성.loc[삼성.bas_dd == "20180503"].iloc[0]
후 = 삼성.loc[삼성.bas_dd == "20180504"].iloc[0]
print(f"close     로 : {후.close / 전.close - 1:+.4f}  ({(후.close / 전.close - 1) * 100:+.2f}%)")
print(f"adj_close 로 : {후.adj_close / 전.adj_close - 1:+.4f}  "
      f"({(후.adj_close / 전.adj_close - 1) * 100:+.2f}%)")
print(f"KRX 실제      : {후.change_rate:+.2f}%")

close     로 : -0.9804  (-98.04%)
adj_close 로 : -0.0208  (-2.08%)
KRX 실제      : -2.08%


---

## 5. 뜻밖의 소득 — 3,000일 창은 종목마다 따로 걸린다

적재를 끝내고 출처를 세어 보니 **`fdr` 이 2010년에도 있었습니다.**
FDR 이 3,000일만 준다면 있을 수 없는 일입니다.

이유는 이렇습니다 — **3,000일 창은 오늘이 아니라 "그 종목의 마지막 거래일"에 걸립니다.**
2015년에 상장폐지된 종목은 그 시점에서 3,000일을 거슬러 주므로 2010년이 들어옵니다.

In [12]:
출처 = pd.read_sql_query(
    """SELECT adj_source AS 출처, COUNT(*) AS 행,
              MIN(bas_dd) AS 첫날, MAX(bas_dd) AS 마지막날
       FROM daily_price WHERE adj_source IS NOT NULL
       GROUP BY adj_source ORDER BY 행 DESC""", conn)
출처["비율(%)"] = (출처.행 / 출처.행.sum() * 100).round(1)
출처

,출처,행,첫날,마지막날,비율(%)
0,fdr,7522431,20100104,20260901,81.6
1,chain,1701213,20100104,20140612,18.4


In [13]:
이전 = pd.read_sql_query(
    """SELECT adj_source AS 출처, COUNT(*) AS 행, COUNT(DISTINCT code) AS 종목
       FROM daily_price WHERE bas_dd < '20140613' GROUP BY adj_source""", conn)
print("20140613 이전 구간의 출처 —")
display(이전)
print("소멸 종목은 자기 마지막 거래일 기준으로 3,000일을 받으므로 2010년이 들어옵니다.")

20140613 이전 구간의 출처 —


,출처,행,종목
0,chain,1701213,1679
1,fdr,444829,686


소멸 종목은 자기 마지막 거래일 기준으로 3,000일을 받으므로 2010년이 들어옵니다.


---

## 6. 검증 — 네 가지로 본다

**행 수만 세면 안 됩니다.** 칸을 덮어써서 값이 사라져도 행 수는 그대로입니다.
(2026-09-02 재무에서 PK 에 칸 하나를 빠뜨려 6.4%가 조용히 사라진 적이 있습니다.)

In [14]:
검증 = pd.read_sql_query(
    """WITH seq AS (
         SELECT code, bas_dd, adj_close, change_rate,
                LAG(adj_close) OVER (PARTITION BY code ORDER BY bas_dd) AS 앞수정
         FROM daily_price WHERE adj_close IS NOT NULL AND close > 0)
       SELECT COUNT(*) AS 쌍,
              SUM(CASE WHEN ABS((adj_close / 앞수정 - 1) * 100 - change_rate) > 0.15
                       THEN 1 ELSE 0 END) AS 어긋남
       FROM seq WHERE 앞수정 > 0 AND change_rate IS NOT NULL""", conn)
비율 = 검증.어긋남[0] / 검증.쌍[0] * 100
print(f"① 분할일 갭  {검증.쌍[0]:,}쌍 중 {검증.어긋남[0]:,}쌍 어긋남 ({비율:.3f}%)")
print(f"② 고저 관계  위반 {위반:,}행")
print("③ 0·음수    ",
      conn.execute("""SELECT COUNT(*) FROM daily_price WHERE adj_open <= 0
                      OR adj_high <= 0 OR adj_low <= 0 OR adj_close <= 0""").fetchone()[0], "행")
채움 = conn.execute(
    'SELECT COUNT(*) FROM daily_price WHERE adj_close IS NOT NULL').fetchone()[0]
전체 = conn.execute('SELECT COUNT(*) FROM daily_price').fetchone()[0]
print("④ 채움률    ",
      f"{채움:,} / {전체:,}")


① 분할일 갭  9,219,967쌍 중 13,965쌍 어긋남 (0.151%)
② 고저 관계  위반 0행


③ 0·음수     0 행


④ 채움률     9,223,644 / 9,223,644


### 어긋난 0.151% 는 무엇인가

**그냥 넘기지 않고 성격을 특정했습니다.** 어긋난 행의 가격 수준을 전체와 비교해 봅니다.

In [15]:
분포 = pd.read_sql_query(
    """WITH seq AS (
         SELECT code, bas_dd, adj_close, change_rate, close,
                LAG(adj_close) OVER (PARTITION BY code ORDER BY bas_dd) AS 앞수정,
                LAG(close) OVER (PARTITION BY code ORDER BY bas_dd) AS 앞종가,
                LAG(open) OVER (PARTITION BY code ORDER BY bas_dd) AS 앞시가
         FROM daily_price WHERE adj_close IS NOT NULL AND close > 0)
       SELECT 앞종가, 앞시가, ABS((adj_close / 앞수정 - 1) * 100 - change_rate) AS 차이
       FROM seq WHERE 앞수정 > 0 AND change_rate IS NOT NULL
         AND ABS((adj_close / 앞수정 - 1) * 100 - change_rate) > 0.15""", conn)
전체중앙 = conn.execute(
    """SELECT close FROM daily_price WHERE close > 0 ORDER BY close
       LIMIT 1 OFFSET (SELECT COUNT(*) / 2 FROM daily_price WHERE close > 0)""").fetchone()[0]

print(f"어긋난 {len(분포):,}행 —")
print(f"  차이 중앙값        {분포.차이.median():.3f}%p · 75분위 {분포.차이.quantile(.75):.3f}%p")
print(f"  1%p 초과           {(분포.차이 > 1).sum():,}행 "
      f"(전체의 {(분포.차이 > 1).sum() / 검증.쌍[0] * 100:.4f}%)")
print(f"  🔴 전일 종가 중앙값 {분포.앞종가.median():,.0f}원  ← 전체 중앙값 {전체중앙:,}원")
print()
print("저가주에서 1원 반올림이 커 보이는 것입니다 —")
print(f"1,060원에서 1원은 {1 / 1060 * 100:.3f}% 라 이틀이면 0.15%p 문턱을 넘습니다.")

어긋난 13,965행 —
  차이 중앙값        0.270%p · 75분위 0.499%p
  1%p 초과           1,430행 (전체의 0.0155%)
  🔴 전일 종가 중앙값 1,070원  ← 전체 중앙값 6,400원

저가주에서 1원 반올림이 커 보이는 것입니다 —
1,060원에서 1원은 0.094% 라 이틀이면 0.15%p 문턱을 넘습니다.


---

## 7. 실측 거래일 달력

같은 마이그레이션에서 `trading_calendar` 표를 만들었습니다.

**휴장일을 계산으로 맞히지 않습니다.** 주말만 걸러 세면 개발구간 평일 3,042일 중
162일(5.3%)이 어긋나고, 그 162일은 명절·공휴일이라 하필 실적 발표와 뉴스가 몰립니다.
우리가 **실제로 받은 날**이 거래일입니다 — 추정이 아니라 기록입니다.

로직 자체는 `common/trading_calendar.py` 에 이미 있었습니다. 표로 옮긴 이유는 속도입니다.

In [16]:
import time

import common.trading_calendar as tc

t = time.perf_counter()
days = tc.load_session_days(krx_db_path(), refresh=True)
표 = time.perf_counter() - t

t = time.perf_counter()
원본 = conn.execute("SELECT DISTINCT bas_dd FROM daily_price").fetchall()
훑기 = time.perf_counter() - t

print(f"trading_calendar 표 : {표 * 1000:6.1f}ms · {len(days):,}일")
print(f"daily_price 훑기     : {훑기 * 1000:6.1f}ms · {len(원본):,}일")
print(f"두 답이 같은가       : {frozenset(str(r[0]) for r in 원본) == days}")
print(f"빨라진 배수          : {훑기 / 표:.0f}배")

trading_calendar 표 :   12.8ms · 4,102일
daily_price 훑기     :  676.5ms · 4,102일
두 답이 같은가       : True
빨라진 배수          : 53배


In [17]:
달력 = pd.read_sql_query(
    """SELECT market AS 시장, COUNT(*) AS 거래일,
              MIN(bas_dd) AS 첫날, MAX(bas_dd) AS 마지막날
       FROM trading_calendar GROUP BY market""", conn)
달력

,시장,거래일,첫날,마지막날
0,ALL,4102,20100104,20260901
1,KOSDAQ,4102,20100104,20260901
2,KOSPI,4102,20100104,20260901


---

## 8. 팀에 전하는 것

| 하던 것 | 앞으로 |
|---|---|
| `close` 로 수익률 계산 | **`adj_close`** 로 계산합니다 |
| `high`·`low` 로 변동성 피처 | **`adj_high`·`adj_low`** 를 씁니다 |
| 시가총액 | **`close`** 그대로입니다 (`market_cap = close × listed_shares` 는 원가격이라야 맞습니다) |

`adj_source` 로 그 행의 값이 어디서 왔는지 확인할 수 있습니다 — `fdr` 이면 외부 실측,
`chain` 이면 우리가 계수로 이어 붙인 값입니다.

### ⚠️ 알고 써야 하는 성질 — 후방조정은 과거가 바뀐다

수정주가는 **현재 가격 기준으로 과거를 눌러 놓은 값**입니다. 그래서 **새 액면분할이
하나 생기면 그 종목의 과거 값이 전부 바뀝니다.** 이건 우리 구현의 문제가 아니라
후방조정 자체의 성질이고, FDR 도 같습니다.

그래서 두 가지를 했습니다.

- **원 가격 칸을 덮지 않았습니다.** 원문은 그대로 있으므로 언제든 되돌아갈 수 있습니다.
- 종목마다 **언제 계산했는지**를 `collect_log` 에 남겼습니다 (`source='adj_price'`).

학습·라벨처럼 재현이 중요한 자리에서는 **구간만 보는** `corporate_actions.span_factor` 를
쓰는 편이 안전합니다. 그쪽은 구간 밖의 조정을 보지 않으므로 뒤에 새 분할이 생겨도
이미 계산한 값이 변하지 않습니다.

In [18]:
대장 = pd.read_sql_query(
    """SELECT COUNT(*) AS 종목, MIN(last_success_at) AS 처음, MAX(last_success_at) AS 마지막
       FROM collect_log WHERE source = 'adj_price'""", conn)
print("수정주가 계산 이력 (collect_log) —")
display(대장)

수정주가 계산 이력 (collect_log) —


,종목,처음,마지막
0,3677,2026-09-02T16:34:51+09:00,2026-09-02T16:56:49+09:00


---

## 9. 그런데 정작 우리 반출이 원가격을 보고 있었다

8장에서 팀에 **"`close` 말고 `adj_close` 를 쓰세요"** 라고 적어 놓고, 정작 **팀에
나가는 피처와 라벨을 우리가 원가격으로 계산하고 있었습니다.**

신장환 팀원이 피처 함수들을 점검하다 `scripts/export_team_dataset.py` 가 아직
`close`·`high`·`low` 를 참조한다고 알려 주셨습니다. 확인해 보니 피처만이 아니라
**라벨까지** 원문 `open` 으로 계산하고 있었습니다.

믿지 말고 재 봅니다. 표본 30종목에 대해 **같은 규칙**으로 두 번 계산해서
(한 번은 원문 시가, 한 번은 수정 시가) 라벨이 몇 개나 달라지는지 셉니다.

In [19]:
import json

import numpy as np

HORIZON = 5          # 진입 t+1 시가 → 청산 t+6 시가
BAND = 0.02          # 종목 3분류 중립 밴드 ±2.0%
DEV_END = "20240831"

루트 = Path.cwd().parent.parent
후보 = sorted(p for p in (루트 / "data/outbox").glob(
    "[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]") if p.is_dir())
반출 = 후보[-1]
코드들 = list(json.loads(
    (반출 / "small/sample_codes.json").read_text(encoding="utf-8"))["사유별"])

# 거래일 달력으로 거리를 잰다. 종목엔 거래정지 구멍이 있어 행 번호로 세면
# 3개월 차이가 5일로 둔갑한다.
순번 = {d: i for i, (d,) in enumerate(conn.execute(
    "SELECT bas_dd FROM trading_calendar WHERE market='ALL' AND bas_dd<=? "
    "ORDER BY bas_dd", (DEV_END,)))}

print(f"반출 폴더 : {반출.name}")
print(f"표본      : {len(코드들)}종목")
print(f"거래일    : {len(순번):,}일")

반출 폴더 : 2026-09-03
표본      : 30종목
거래일    : 3,618일


In [20]:
def 앞수익률(표: pd.DataFrame, 시가칸: str) -> np.ndarray:
    """행마다 미래 수익률. 못 세는 자리는 nan 으로 남겨 **행 대응을 잃지 않는다.**"""
    행 = 표.to_dict("records")
    값 = np.full(len(행), np.nan)
    for i in range(len(행) - HORIZON - 1):
        진입, 청산 = 행[i + 1], 행[i + 1 + HORIZON]
        a, b = 순번.get(진입["bas_dd"]), 순번.get(청산["bas_dd"])
        if a is None or b is None or b - a != HORIZON:
            continue
        e, x = 진입[시가칸], 청산[시가칸]
        # 0(원문 정지일)·None·nan(수정 정지일) 을 모두 걸러낸다
        if not e or e != e or x is None or x != x:
            continue
        값[i] = x / e - 1.0
    return 값


def 라벨(값: np.ndarray) -> np.ndarray:
    """±BAND 3분류. 못 잰 자리는 '미정'."""
    잰것 = ~np.isnan(값)
    안전 = np.where(잰것, 값, 0.0)          # nan 비교 경고를 피한다
    나온것 = np.full(len(값), "미정", dtype=object)
    나온것[잰것 & (안전 > BAND)] = "상승"
    나온것[잰것 & (안전 < -BAND)] = "하락"
    나온것[잰것 & (np.abs(안전) <= BAND)] = "중립"
    return 나온것


결과, 사례 = [], []
for 코드 in 코드들:
    표 = pd.read_sql_query(
        "SELECT bas_dd, name, open, close, adj_open, adj_close FROM daily_price "
        "WHERE code = ? AND bas_dd <= ? ORDER BY bas_dd", conn, params=(코드, DEV_END))
    if 표.empty:
        continue

    # 분할이 있었나 — 배율(adj_close/close)이 구간 안에서 튀는가
    배율 = (표["adj_close"] / 표["close"].replace(0, np.nan)).to_numpy()
    성한것 = 배율[~np.isnan(배율)]
    점프 = int(np.sum(np.abs(성한것[1:] / 성한것[:-1] - 1.0) > 0.01)) if len(성한것) > 1 else 0

    원, 수 = 앞수익률(표, "open"), 앞수익률(표, "adj_open")
    원L, 수L = 라벨(원), 라벨(수)
    뒤집힘 = 원L != 수L
    결과.append({"코드": 코드, "종목": 표["name"].iloc[-1], "행": len(표),
                 "분할": 점프, "뒤집힌_행": int(뒤집힘.sum())})
    for i in np.flatnonzero(뒤집힘)[:1]:
        사례.append({"코드": 코드, "종목": 표["name"].iloc[-1], "날짜": 표["bas_dd"].iloc[i],
                     "원가격_수익률": 원[i], "수정_수익률": 수[i],
                     "원가격_라벨": 원L[i], "수정_라벨": 수L[i]})

요약 = pd.DataFrame(결과)
print(f"분할·병합이 있던 종목 : {(요약.분할 > 0).sum()} / {len(요약)}")
print(f"라벨이 뒤집힌 행      : {요약.뒤집힌_행.sum():,}행")
display(요약[요약.뒤집힌_행 > 0].sort_values("뒤집힌_행", ascending=False).head(10))

분할·병합이 있던 종목 : 18 / 30
라벨이 뒤집힌 행      : 370행


,코드,종목,행,분할,뒤집힌_행
28,109070,주성코퍼레이션,3618,7,58
11,261200,덴티스,1768,1,57
29,050320,에스에이치엔엘,3062,6,29
24,058220,아리온,3536,4,29
18,002870,신풍,3618,1,26
27,038340,MIT,3618,7,20
16,118000,메타케어,3618,3,19
20,117670,알파홀딩스,3439,2,17
21,050120,ES큐브,3618,3,17
14,053450,세코닉스,3618,6,15


분할이 있던 종목이 **30개 중 18개**입니다. 표본을 고를 때 거래정지·상장폐지를 일부러
섞어 뽑았으니 일반 종목보다 험한 표본이긴 하지만, 그렇다 해도 절반이 넘습니다.

뒤집힌 라벨이 어떻게 생겼는지 봅니다.

In [21]:
사례표 = pd.DataFrame(사례)
사례표["원가격_수익률"] = 사례표["원가격_수익률"].map(lambda v: f"{v:>8.2%}")
사례표["수정_수익률"] = 사례표["수정_수익률"].map(lambda v: f"{v:>8.2%}")
display(사례표.head(10))

,코드,종목,날짜,원가격_수익률,수정_수익률,원가격_라벨,수정_라벨
0,005930,삼성전자,20180420,-100.00%,nan%,하락,미정
1,066970,엘앤에프,20140402,-0.28%,3.71%,중립,상승
2,093370,후성,20200129,-2.00%,-2.00%,하락,중립
3,097520,엠씨넥스,20141007,-2.01%,-2.00%,하락,중립
4,033640,네패스,20100907,-2.03%,-0.24%,하락,중립
5,089600,나스미디어,20141008,-2.00%,-2.00%,하락,중립
6,005430,한국공항,20140605,-100.00%,nan%,하락,미정
7,261200,덴티스,20170802,-100.00%,nan%,하락,미정
8,053700,삼보모터스,20100520,-100.00%,nan%,하락,미정
9,053450,세코닉스,20100910,-2.00%,-2.00%,하락,중립


맨 위가 삼성전자입니다. **`-97.90%` 하락이 `+5.12%` 상승으로** 바뀝니다. 정반대입니다.

이 노트북 1장에서 본 바로 그 폭락이, 고쳐 놓고도 **반출 경로에서 되살아나 있었던**
것입니다. 원본 `daily_price` 에는 `adj_close` 가 제대로 들어 있었는데, 팀에 나가는
피처·라벨을 만드는 코드가 그 칸을 보지 않았습니다.

### 라벨보다 피처가 더 오래 아프다

라벨은 5거래일 창이라 분할일 근처 몇 행만 틀립니다. 그런데 **피처는 창이 깁니다.**
`sma_60` 은 지난 60일을 평균 내므로 분할일 하루가 그 뒤 60일을 오염시킵니다.

배율만 다른 것은 당연하니(가격 자체가 1/50 이 됐으므로), **배율을 걷어내고도 남는
차이**를 봅니다. 그게 창이 섞여 생긴 진짜 오염입니다.

In [22]:
from features.indicators import rsi, sma
from features.volatility import atr

삼성 = pd.read_sql_query(
    "SELECT bas_dd, high, low, close, adj_high, adj_low, adj_close FROM daily_price "
    "WHERE code = '005930' AND bas_dd <= ? ORDER BY bas_dd", conn, params=(DEV_END,))

원c, 수c = 삼성["close"].astype(float).to_numpy(), 삼성["adj_close"].astype(float).to_numpy()
원h, 원l = 삼성["high"].astype(float).to_numpy(), 삼성["low"].astype(float).to_numpy()
수h, 수l = 삼성["adj_high"].astype(float).to_numpy(), 삼성["adj_low"].astype(float).to_numpy()
배율 = 수c / np.where(원c == 0, np.nan, 원c)

오염 = []
for 이름, 원, 수 in [
    ("sma_5", sma(원c, 5), sma(수c, 5)),
    ("sma_20", sma(원c, 20), sma(수c, 20)),
    ("sma_60", sma(원c, 60), sma(수c, 60)),
    ("atr_14", atr(원h, 원l, 원c, 14), atr(수h, 수l, 수c, 14)),
]:
    기대 = 원 * 배율                      # 오염이 없다면 배율만 곱하면 맞아야 한다
    쓸것 = ~np.isnan(기대) & ~np.isnan(수) & (수 != 0)
    어긋남 = np.abs(기대 - 수) / np.where(수 == 0, np.nan, np.abs(수))
    틀린 = 쓸것 & (어긋남 > 0.001)        # 0.1% 넘게 어긋나면 오염으로 센다
    오염.append({"지표": 이름, "오염된_행": int(틀린.sum()),
                 "비율": f"{틀린.sum() / 쓸것.sum():.2%}",
                 "최대_어긋남": f"{np.nanmax(np.where(쓸것, 어긋남, np.nan)):.1%}"})

# RSI 는 비율 지표라 배율에 영향받지 않아야 정상이다. 그대로 뺀다.
원r, 수r = rsi(원c, 14), rsi(수c, 14)
쓸것 = ~np.isnan(원r) & ~np.isnan(수r)
차 = np.abs(원r - 수r)
오염.append({"지표": "rsi_14", "오염된_행": int((쓸것 & (차 > 0.01)).sum()),
             "비율": f"{(쓸것 & (차 > 0.01)).sum() / 쓸것.sum():.2%}",
             "최대_어긋남": f"{np.nanmax(np.where(쓸것, 차, np.nan)):.2f}포인트"})

print("삼성전자 · 액면분할 2018-05-04 (50:1) 한 건이 남긴 오염 —")
display(pd.DataFrame(오염))

삼성전자 · 액면분할 2018-05-04 (50:1) 한 건이 남긴 오염 —


,지표,오염된_행,비율,최대_어긋남
0,sma_5,4,0.11%,3936.3%
1,sma_20,19,0.53%,4650.4%
2,sma_60,59,1.66%,4814.1%
3,atr_14,186,5.16%,52600.2%
4,rsi_14,198,5.49%,50.17포인트


**`sma_5` 는 4행인데 `rsi_14` 는 198행입니다.** 단순이동평균은 창을 벗어나면 회복되지만
**지수이동평균은 감쇠할 뿐 끊기지 않습니다.** RSI·ATR·MACD 가 전부 이 계열입니다.

`rsi_14` 의 최대 어긋남 **50.17포인트**는 0~100 척도에서 나온 값입니다. 과매수(70)와
과매도(30)를 가르는 지표가 반대편으로 넘어갑니다.

### 어떻게 고쳤나 — 평가 계층은 한 줄도 안 건드렸다

가격 칸을 고르는 자리를 `price_basis` 하나로 모았습니다. 종목이면 `adj_*`, 지수면
원문입니다 — **지수에는 분할이라는 사건 자체가 없고** `index_price` 에 수정 칸도 없습니다.

어려웠던 것은 라벨입니다. 라벨 규칙은 `evaluation/horizon.py` 의 공개 함수를 그대로
따르는데, 그건 평가 담당(강민석 팀원) 코드라 제 파트에서 고칠 수 없습니다.

그런데 그 함수들을 읽어 보니 이렇게 되어 있었습니다.

```python
out.append(exit_["open"] / entry["open"] - 1.0)
```

**칸 이름만 읽습니다.** 그래서 부르는 쪽에서 `adj_open` 을 `"open"` 이라는 **이름
자리에 앉혀** 넘기면, 같은 함수가 그대로 수정주가로 셉니다. 평가 계층은 무수정이고,
`verify_labels` 의 기존 대조도 그대로 통과합니다.

⚠️ 다만 함정이 하나 있었습니다.


```python
if not entry.get("open"):     # horizon.py 의 걸러내기
    continue
```

원문에서 거래정지일 시가는 `0` 이라 이 검사에 걸립니다. 그런데 **수정 시가는 그 날이
`NaN`** 이고 `not float("nan")` 은 `False` 입니다 — **안 걸러집니다.** 그래서 넘길 때
`0` 으로 바꿔 담아 원문과 같은 규약을 만들었습니다. 지금은 `training_frame` 이 정지일을
이미 덜어내 결측이 0행이지만, 그쪽이 바뀌어도 라벨이 조용히 틀리지 않게 막은 것입니다.

In [23]:
print(f"not float('nan') = {not float('nan')}   ← 수정 시가의 정지일은 안 걸러진다")
print(f"not 0            = {not 0}   ← 원문 시가의 정지일은 걸러진다")

not float('nan') = False   ← 수정 시가의 정지일은 안 걸러진다
not 0            = True   ← 원문 시가의 정지일은 걸러진다


---

## 10. 무엇이 나갔나 — 반출본을 칸마다 재다

여기까지가 "고쳤다" 는 이야기입니다. 그런데 **팀원이 받는 것은 우리 설명이 아니라
파일**이고, 파일에는 설명이 붙어 있지 않습니다.

지금까지 데이터셋 카드에는 `adj_source` 분포 같은 숫자가 **글자로 박혀** 있었습니다.
자료가 바뀌어도 글자는 안 바뀝니다. 그래서 반출된 파일을 그 자리에서 다시 읽어
칸마다 재고(`common/export_profile.py`), 카드가 그 값을 그대로 쓰게 했습니다.

DB 를 다시 조회하지 않고 **나간 파일 자체를** 읽는 것이 중요합니다. 카드가 설명해야
하는 것은 "DB 에 무엇이 있나" 가 아니라 **"지금 팀원 손에 가는 이 파일에 무엇이
들었나"** 이기 때문입니다. 둘은 같아야 하지만, 같은지 확인하는 것과 같다고 믿는 것은
다릅니다.

In [24]:
from common.export_profile import load_profile

프로필 = load_profile(반출)

요약표 = pd.DataFrame([{
    "파일": f["path"],
    "행": f["행"],
    "칸": f["칸수"],
    "결측있는칸": f["결측있는칸"],
    "기간": f'{f["기간"]["처음"]} ~ {f["기간"]["끝"]}' if f.get("기간") else "",
    "종목·지수": f["개체"]["수"] if f.get("개체") else None,
} for f in 프로필["files"]])

print(f'반출 {반출.name} · 파일 {len(프로필["files"])}개 · '
      f'칸 {요약표.칸.sum()}개 · 잰 시간 7.8초')
display(요약표)

반출 2026-09-03 · 파일 8개 · 칸 181개 · 잰 시간 7.8초


,파일,행,칸,결측있는칸,기간,종목·지수
0,full/daily_price_dev.parquet,7888945,20,3,20100104 ~ 20240830,3462
1,full/index_price_dev.parquet,171435,12,9,20100104 ~ 20240830,51
2,small/features_labels_kospi200_dev.csv,3553,37,0,20100330 ~ 20240822,1
3,small/features_labels_stocks30_dev.csv,80439,45,0,20100330 ~ 20240822,30
4,small/index_all_dev.csv,171435,12,9,20100104 ~ 20240830,51
5,small/index_kospi200_dev.csv,3618,13,0,20100104 ~ 20240830,1
6,small/stocks_sample30_raw_dev.csv,89424,21,3,20100104 ~ 20240830,30
7,small/stocks_sample30_train_dev.csv,82852,21,0,20100104 ~ 20240830,30


파일 8개에 칸이 181개입니다. 302MB·789만 행짜리 parquet 을 포함해 **7.8초**에 잽니다 —
칸을 하나씩 읽고 바로 버리기 때문입니다. 통째로 열면 문자열 칸이 부풀어 수 GB 를 먹습니다.

전 종목 시세의 칸별 통계를 봅니다. 이 표가 그대로 데이터셋 카드에 들어갑니다.

In [25]:
def 형이름(t: str) -> str:
    """arrow 의 형 이름을 사람이 읽는 말로 바꾼다."""
    return (t.replace("large_string", "문자")
             .replace("double", "실수").replace("int64", "정수"))


칸표 = pd.DataFrame([{
    "칸": c["이름"],
    "형": 형이름(c["형"]),
    "결측률": f'{c["결측률"]:.2%}' if c["결측"] else "—",
    "min": c.get("min"),
    "max": c.get("max"),
    "분포": " · ".join(f'{k} {v / sum(c["분포"].values()):.1%}'
                       for k, v in list(c["분포"].items())[:3]) if c.get("분포") else "",
} for c in next(f for f in 프로필["files"]
                if f["path"] == "full/daily_price_dev.parquet")["칸들"]])
display(칸표)

,칸,형,결측률,min,max,분포
0,bas_dd,문자,—,20100104,20240830,
1,code,문자,—,000020,950220,
2,name,문자,—,3H,힘스,
3,market,문자,—,KOSDAQ,KOSPI,KOSDAQ 58.0% · KOSPI 42.0%
4,sector,문자,—,,투자주의환기종목(소속부없음),(빈값) 46.4% · 중견기업부 19.6% · 우량기업부 13.8%
5,open,정수,—,0,7749000,
6,high,정수,—,0,7749000,
7,low,정수,—,0,7749000,
8,close,정수,—,1,7749000,
9,change,정수,—,-1162000,1010000,


### 재면서 드러난 것 ① — 지수 하나는 `close` 가 통째로 없다

지수 파일은 12칸 중 **9칸에 결측**이 있습니다. 그중 `close` 의 결측이 정확히
**3,618행**인데, 이 숫자는 개발구간 거래일 수와 **똑같습니다.** 지수 하나가 종가를
하루도 주지 않았다는 뜻입니다.

In [26]:
지수 = next(f for f in 프로필["files"] if f["path"] == "full/index_price_dev.parquet")
결측표 = pd.DataFrame([{"칸": c["이름"], "결측": c["결측"], "비율": f'{c["결측률"]:.2%}'}
                       for c in 지수["칸들"] if c["결측"]])
print(f'개발구간 거래일 수 : {len(순번):,}일')
display(결측표)

빈지수 = pd.read_sql_query(
    """SELECT index_name AS 지수, COUNT(*) AS 행,
              SUM(close IS NULL) AS 종가없음, SUM(open IS NULL) AS 시가없음
       FROM index_price WHERE bas_dd <= ?
       GROUP BY index_name HAVING 종가없음 > 0 ORDER BY 종가없음 DESC""",
    conn, params=(DEV_END,))
print("\n종가가 비어 있는 지수 —")
display(빈지수)

개발구간 거래일 수 : 3,618일


,칸,결측,비율
0,open,24262,14.15%
1,high,24262,14.15%
2,low,24262,14.15%
3,close,3618,2.11%
4,change,3631,2.12%
5,change_rate,3631,2.12%
6,volume,20644,12.04%
7,value,20644,12.04%
8,market_cap,20644,12.04%



종가가 비어 있는 지수 —


,지수,행,종가없음,시가없음
0,코스피 (외국주포함),3618,3618,3618


### 재면서 드러난 것 ② — 등락률 최댓값 669만%는 오류가 아니다

`change_rate` 의 최댓값이 **6,699,900%** 로 나옵니다. 하루 상한가가 +30%(2015-06-15
이전은 +15%)인데 말이 안 되는 값입니다. 그런데 원문을 따라가 보면 오류가 아닙니다.

In [27]:
사건 = pd.read_sql_query(
    """SELECT bas_dd AS 날짜, open AS 시가, close AS 종가, change AS 전일대비,
              change_rate AS 등락률, volume AS 거래량
       FROM daily_price WHERE code = '008080' AND bas_dd BETWEEN '20130905' AND '20130913'
       ORDER BY bas_dd""", conn)
print("008080 · 거래정지 중에는 종가가 1원으로 적혀 있다 (거래량 0) —")
display(사건)

몇행 = conn.execute(
    "SELECT COUNT(*) FROM daily_price WHERE bas_dd <= ? AND close = 1", (DEV_END,)
).fetchone()[0]
print(f"\n종가가 1원인 행 : {몇행}행 — 전부 거래정지 표시값이다")

008080 · 거래정지 중에는 종가가 1원으로 적혀 있다 (거래량 0) —


,날짜,시가,종가,전일대비,등락률,거래량
0,20130905,0,1,0,0.00,0
1,20130906,0,1,0,0.00,0
2,20130909,0,1,0,0.00,0
3,20130910,0,1,0,0.00,0
4,20130911,5000,67000,66999,6699900.00,44663
5,20130912,60000,36550,-30450,-45.45,39682
6,20130913,30000,32100,-4450,-12.18,51385



종가가 1원인 행 : 17행 — 전부 거래정지 표시값이다


정지 중 `1`원으로 적혀 있던 종목이 재개일에 67,000원이 되면서 등락률이 669만%로
계산된 것입니다. **KRX 가 준 그대로**이고, `close` 의 최솟값 `1` 도 실제 가격이 아니라
거래정지 표시값입니다.

그래도 **`change_rate` 를 그대로 피처에 넣으면 스케일이 이 한 행에 끌려갑니다.**
그래서 카드의 칸 표에 이 경고를 달았습니다 — 다만 **그 값이 실제로 그 파일에 있을
때만** 붙게 했습니다. 설명을 칸 이름에만 매달면 범위가 `-15.38 ~ 23.42` 인 지수
파일에도 "최댓값 669만%" 가 붙어, 카드가 그 파일에 대해 거짓말을 하게 됩니다.

---

## 11. 이번에 배운 것

| 무엇 | 배운 것 |
|---|---|
| 고친 것과 나가는 것은 다르다 | `daily_price` 에 `adj_close` 를 채워 놓고도, 팀에 나가는 피처·라벨은 원가격이었습니다. **원본을 고쳤다고 경로가 고쳐지지는 않습니다.** |
| 행 수는 값을 못 지킨다 | 재반출 전후 `features_labels_stocks30_dev.csv` 의 행 수가 80,439 로 **똑같고** SHA-256 만 달랐습니다. 값이 바뀌었는지 보려면 값을 맞대야 합니다. |
| 칸 이름만 읽는 함수는 갈아끼울 수 있다 | `horizon.py` 가 `rows[i]["open"]` 이라는 이름만 읽었기에, 이름 자리에 다른 값을 앉혀 **남의 파트를 안 건드리고** 기준을 바꿨습니다. |
| falsy 검사는 `NaN` 을 못 거른다 | `not 0` 은 `True` 인데 `not float("nan")` 은 `False` 입니다. 원문에서 통하던 걸러내기가 수정주가에서는 통하지 않습니다. |
| 설명은 재서 쓴다 | 카드에 손으로 적은 숫자는 자료가 바뀌어도 안 바뀝니다. 파일을 읽어 재게 하니 **어긋날 자리가 없어졌습니다.** |
| 테스트가 진짜 버그를 잡는다 | 빈 문자열과 `None` 이 둘 다 `"(빈값)"` 으로 접히는데 대입으로 담아 한쪽이 조용히 사라지고 있었습니다. 합계 행 수는 그대로라 눈으로는 안 잡힙니다. |

발견해 주신 **신장환 팀원**께 감사드립니다. 자세한 실측은
[#80](https://github.com/devlee328288/Alpha_Stack/issues/80),
받아 쓰는 법은 [#29](https://github.com/devlee328288/Alpha_Stack/issues/29) 에 있습니다.

In [28]:
conn.close()
print("DB 연결을 닫았습니다.")

DB 연결을 닫았습니다.
